# Model Evaluation: Regression

## Overview

Regression model evaluation answers two distinct questions:
1. **Fit:** How well does the model describe the training data?
2. **Prediction:** How well does the model generalise to new data?

These require different metrics and different data. Never evaluate generalisation on the same data used for fitting.

| Metric | Formula | Interpretation |
|---|---|---|
| RMSE | sqrt(mean((y - y_hat)^2)) | Average error in y units; penalises large errors |
| MAE | mean(|y - y_hat|) | Average absolute error; robust to outliers |
| R-squared | 1 - SS_res/SS_tot | Proportion of variance explained (training set) |
| Adjusted R2 | penalised R2 | Penalises for number of predictors |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats

rng = np.random.default_rng(42)
n = 200
elevation  = rng.uniform(50, 400, n)
nitrate    = rng.gamma(2, 2, n)
treatment  = rng.choice([0,1], n)
richness   = 28 - 0.035*elevation - 0.8*nitrate + 2.5*treatment + rng.normal(0, 3.5, n)
X = np.column_stack([elevation, nitrate, treatment])
y = richness
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

---
## Training vs Test Metrics

In [ ]:
model = LinearRegression().fit(X_tr, y_tr)
y_tr_pred = model.predict(X_tr)
y_te_pred = model.predict(X_te)
for split, yt, yp in [("Train", y_tr, y_tr_pred),("Test", y_te, y_te_pred)]:
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    print(f"{split}: RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}")

---
## Cross-Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = np.sqrt(-cross_val_score(LinearRegression(), X, y,
                                    cv=kf, scoring="neg_mean_squared_error"))
cv_r2   = cross_val_score(LinearRegression(), X, y, cv=kf, scoring="r2")
print(f"5-fold CV RMSE: {cv_rmse.mean():.3f} +/- {cv_rmse.std():.3f}")
print(f"5-fold CV R2:   {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")

---
## Detecting Overfitting: Complexity vs Error

In [ ]:
degrees = range(1, 8)
tr_rmse, te_rmse = [], []
for d in degrees:
    pipe = Pipeline([
        ("poly", PolynomialFeatures(d, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
    pipe.fit(X_tr[:, :1], y_tr)  # use elevation only for clear illustration
    tr_rmse.append(np.sqrt(mean_squared_error(y_tr, pipe.predict(X_tr[:,:1]))))
    te_rmse.append(np.sqrt(mean_squared_error(y_te, pipe.predict(X_te[:,:1]))))
plt.figure(figsize=(7,4))
plt.plot(degrees, tr_rmse, "o-", color="steelblue", label="Train RMSE")
plt.plot(degrees, te_rmse, "o-", color="#e74c3c", label="Test RMSE")
plt.xlabel("Polynomial degree"); plt.ylabel("RMSE")
plt.title("Bias-Variance Tradeoff: Train vs Test Error by Model Complexity")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Best test RMSE at degree {np.argmin(te_rmse)+1}")

---
## Residual Analysis

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4))
resid_te = y_te - y_te_pred
axes[0].scatter(y_te_pred, resid_te, alpha=0.5, s=20, color="steelblue")
axes[0].axhline(0, color="#e74c3c", lw=1.5)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Predicted (test set)")
axes[1].scatter(y_te, y_te_pred, alpha=0.5, s=20, color="steelblue")
axes[1].plot([y.min(),y.max()],[y.min(),y.max()],"--",color="#e74c3c",lw=1.5)
axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
axes[1].set_title("Actual vs Predicted")
(osm,osr),(sl,ic,r) = stats.probplot(resid_te, dist="norm")
axes[2].scatter(osm, osr, s=15, alpha=0.5, color="steelblue")
axes[2].plot(osm,sl*np.array(osm)+ic,color="#e74c3c",lw=1.5)
axes[2].set_title(f"Q-Q Residuals (r={r:.3f})")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Evaluating model performance on training data only**  
Training metrics always improve as model complexity increases, even when adding noise predictors. Always evaluate on a held-out test set or via cross-validation to assess generalisation.

**2. Using R-squared to compare models with different numbers of predictors**  
R-squared never decreases when adding predictors, even irrelevant ones. Use adjusted R-squared for within-sample comparison or cross-validated R-squared for generalisation assessment.

**3. Choosing the best model by test RMSE and then reporting that RMSE as the model's performance**  
Once you use the test set to select a model, that RMSE is optimistic. Reserve the test set for a single final evaluation. Use a validation set or nested CV for model selection.

**4. Not plotting residuals on the test set**  
A model with acceptable RMSE may still have systematic bias: residuals increasing with fitted values, non-linear patterns, or outliers. Always plot residuals vs predicted on the test set.

**5. Treating RMSE as interpretable without checking its scale against y**  
RMSE=3.5 is informative only relative to the range and typical values of y. Report RMSE alongside the mean and SD of the outcome so readers can judge whether the error is practically acceptable.
---
*python_methods_library - Samantha McGarrigle*